# Preprocessing and Translation

In [32]:
import re
import emoji
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException
from tqdm import tqdm
import time
import numpy as np
import pandas as pd

## Load CSV file

In [25]:
# Load your CSV file
# CHANGE THIS PATH to your actual CSV file location
df = pd.read_csv('trendingtopic.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\n First few rows:")
df.head(10)

Dataset Shape: (10595, 8)
Columns: ['id', 'text', 'created_at', 'username', 'likes', 'retweets', 'topic', 'timestamp']

 First few rows:


,id,text,created_at,username,likes,retweets,topic,timestamp
0,1960875468531621946,@PsyGuy007 bien vue super les gars...,2025-08-28,JaegerStep78492,0.0,NaN,Super Bowl LX,NaN
1,1960973143008292871,"You know what, I've ALSO been a splendid nigga...",2025-08-28,JSRingo,209.0,NaN,Super Bowl LX,NaN
2,1961159089544990909,@fashion_nfl Don’t think enough people are tal...,2025-08-28,p23rce,479.0,NaN,Super Bowl LX,NaN
3,1960932935676387741,RT @tahsin_adib: NINGNING telling winter it mu...,2025-08-28,hidden_faayz,0.0,NaN,Winter Olympics 2026 Milano,NaN
4,1961193691605078397,🤚🤚🤚💙2026!!!,2025-08-28,defloody,1.0,NaN,Winter Olympics 2026 Milano,NaN
5,1961003661548741029,Super sleepy. https://t.co/FMEc5ShIuv,2025-08-28,FajitaFantom,17.0,NaN,Super Bowl LX,NaN
6,1961078342188240986,@FabrizioRomano Check $RIZZ out guysWe're stil...,2025-08-28,Nazariite,3.0,NaN,Super Bowl LX,NaN
7,1960861972188033264,Nvidia (NVDA) earnings report Q2 2026 https://...,2025-08-28,KWoljevach71778,54.0,NaN,Winter Olympics 2026 Milano,NaN
8,1961104466297659629,“Republicans against Trump”Supports democrat w...,2025-08-28,mesnickerlicker,28.0,NaN,Super Bowl LX,NaN
9,1961169372170309929,RT @laymagdalene333: not super surprising but ...,2025-08-28,rosyghoul,0.0,NaN,Super Bowl LX,NaN


## Change category name
- from "Bangladeshi Election 2026" to "Politics"

In [27]:
# remove extra spaces first (important because your value has trailing spaces)
df['topic'] = df['topic'].str.strip()

# replace the specific category
df['topic'] = df['topic'].replace("Bangladesh Election 2026", "Politics")

# optional: check results
print(df['topic'].unique())

<StringArray>
['Super Bowl LX', 'Winter Olympics 2026 Milano', 'Politics']
Length: 3, dtype: str


## Remove duplicate posts

In [28]:
# Check for duplicates
duplicates_id = df.duplicated(subset='id').sum()
duplicates_text = df.duplicated(subset='text').sum()
print(f"Duplicate IDs: {duplicates_id}")
print(f"Duplicate Texts: {duplicates_text} ({duplicates_text/len(df)*100:.2f}%)")

Duplicate IDs: 1
Duplicate Texts: 277 (2.61%)


In [29]:
# Remove duplicates based on text
before = len(df)
df = df.drop_duplicates(subset='text')
print(f"Removed {before - len(df)} duplicate tweets")
print(f"New dataset shape: {df.shape}")

Removed 277 duplicate tweets
New dataset shape: (10318, 8)


## Decide whether to keep retweets

In [ ]:
# Extract the actual content from RTs (remove "RT @user:" prefix)
def extract_rt_content(text):
    if text.startswith('RT @'):
        # Remove "RT @username: " to get the actual content
        return re.sub(r'^RT @\w+:\s*', '', text)
    return text

# Get RT contents
rt_texts = df[df['text'].str.startswith('RT @', na=False)]['text'].apply(extract_rt_content)

# Get original texts
original_texts = df[~df['text'].str.startswith('RT @', na=False)]['text']

# Check if RT content exists in originals
matches = 0
for rt_text in rt_texts:
    # Check if this RT content appears in any original tweet
    if any(rt_text in orig for orig in original_texts):
        matches += 1

match_percentage = (matches / len(rt_texts)) * 100 if len(rt_texts) > 0 else 0

print(f"\n🔍 DETAILED VERIFICATION:")
print(f"Original tweets: {len(original_texts)}")
print(f"Retweets: {len(rt_texts)}")
print(f"RTs that match originals: {matches} ({match_percentage:.1f}%)")

if match_percentage > 50:
    print("\n✅ DECISION: REMOVE retweets (originals are in dataset)")
else:
    print("\n✅ DECISION: KEEP retweets (originals are missing)")


🔍 DETAILED VERIFICATION:
Original tweets: 8266
Retweets: 2052
RTs that match originals: 5 (0.2%)

✅ DECISION: KEEP retweets (originals are missing)


In [31]:
!pip install emoji

   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 608.4/608.4 kB 4.4 MB/s  0:00:00


## Clean text 

In [33]:
def clean_text(text):
    """
    Clean tweet text: remove URLs, mentions, RT prefix, emojis.
    
    Parameters:
    -----------
    text : str
        Raw tweet text
        
    Returns:
    --------
    str
        Cleaned text
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove RT prefix
    text = re.sub(r'^RT\s+@\w+:\s*', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    
    # Remove emojis
    text = emoji.replace_emoji(text, replace='')
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text.strip()

def clean_dataframe(df, text_column='text'):
    """
    Apply cleaning to entire dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing tweets
    text_column : str
        Column to clean (default: 'text_translated')
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with new 'text_cleaned' column
    """
    print("\n🧹 Cleaning text...")
    
    tqdm.pandas(desc="Cleaning")
    df['text_cleaned'] = df[text_column].progress_apply(clean_text)
    
    # Remove empty tweets after cleaning
    before = len(df)
    df = df[df['text_cleaned'].str.len() >= 10].copy()
    removed = before - len(df)
    
    if removed > 0:
        print(f"   ✅ Removed {removed} tweets that became too short after cleaning")
    
    print(f"   ✅ Final dataset: {len(df)} tweets")
    
    return df

In [34]:
cleaned_df = clean_dataframe(df)
cleaned_df.head()


🧹 Cleaning text...


Cleaning: 100%|██████████| 10318/10318 [00:00<00:00, 11347.38it/s]

   ✅ Removed 85 tweets that became too short after cleaning
   ✅ Final dataset: 10233 tweets


,id,text,created_at,username,likes,retweets,topic,timestamp,text_cleaned
0,1960875468531621946,@PsyGuy007 bien vue super les gars...,2025-08-28,JaegerStep78492,0.0,NaN,Super Bowl LX,NaN,bien vue super les gars...
1,1960973143008292871,"You know what, I've ALSO been a splendid nigga...",2025-08-28,JSRingo,209.0,NaN,Super Bowl LX,NaN,"You know what, I've ALSO been a splendid nigga..."
2,1961159089544990909,@fashion_nfl Don’t think enough people are tal...,2025-08-28,p23rce,479.0,NaN,Super Bowl LX,NaN,Don’t think enough people are talking about th...
3,1960932935676387741,RT @tahsin_adib: NINGNING telling winter it mu...,2025-08-28,hidden_faayz,0.0,NaN,Winter Olympics 2026 Milano,NaN,NINGNING telling winter it must've take a long...
5,1961003661548741029,Super sleepy. https://t.co/FMEc5ShIuv,2025-08-28,FajitaFantom,17.0,NaN,Super Bowl LX,NaN,Super sleepy.


## Translation

In [36]:
def detect_languages(df, text_column='text_cleaned', sample_size=None):
    """
    Detect language for each tweet in the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing tweets
    text_column : str
        Name of the column containing text (default: 'text')
    sample_size : int or None
        If provided, only detect language for a random sample (faster)
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with new 'detected_lang' column
    """
    print("🔍 Detecting languages...")
    
    # Initialize column
    df['detected_lang'] = 'en'
    
    # Determine which rows to check
    if sample_size and sample_size < len(df):
        indices_to_check = df.sample(sample_size, random_state=42).index
        print(f"   (Checking sample of {sample_size} tweets)")
    else:
        indices_to_check = df.index
    
    # Detect languages
    for idx in tqdm(indices_to_check, desc="Detecting languages"):
        try:
            text = str(df.loc[idx, text_column])
            if len(text.strip()) > 10:
                df.loc[idx, 'detected_lang'] = detect(text)
        except LangDetectException:
            df.loc[idx, 'detected_lang'] = 'unknown'
        except Exception:
            pass
    
    # Show distribution
    lang_counts = df['detected_lang'].value_counts()
    print(f"\n📊 Language Distribution:")
    for lang, count in lang_counts.head(10).items():
        print(f"   • {lang}: {count} ({count/len(df)*100:.1f}%)")
    
    non_english = (df['detected_lang'] != 'en').sum()
    print(f"\n⚠️  Non-English tweets: {non_english} ({non_english/len(df)*100:.1f}%)")
    
    return df


# ============================================
# FUNCTION 2: TRANSLATION
# ============================================

def translate_non_english(df, text_column='text_cleaned', lang_column='detected_lang', delay=0.05):
    """
    Translate non-English tweets to English using Google Translate (FREE).
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing tweets and detected languages
    text_column : str
        Name of the column containing text to translate
    lang_column : str
        Name of the column containing detected language
    delay : float
        Delay between API calls in seconds (default: 0.05)
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with new 'text_translated' column
    """
    print("\n🌐 Translating non-English tweets...")
    
    # Initialize column with cleaned text
    df['text_translated'] = df[text_column].copy()
    
    # Find tweets that need translation
    needs_translation = (df[lang_column] != 'en') & (df[lang_column] != 'unknown')
    n_to_translate = needs_translation.sum()
    
    print(f"   Translating {n_to_translate} tweets...")
    
    if n_to_translate == 0:
        print("   ✅ No translation needed - all tweets are English!")
        return df
    
    # Translate
    translator = GoogleTranslator(source='auto', target='en')
    
    for idx in tqdm(df[needs_translation].index, desc="Translating"):
        try:
            original_text = str(df.loc[idx, text_column])
            
            if len(original_text.strip()) > 5:
                translated = translator.translate(original_text)
                df.loc[idx, 'text_translated'] = translated
                
                # Small delay to avoid rate limiting
                time.sleep(delay)
                
        except Exception as e:
            # If translation fails, keep original
            pass
    
    print(f"\n✅ Translation complete!")
    
    # Show examples
    print(f"\n📝 Translation Examples:")
    print("="*80)
    
    translated_samples = df[needs_translation].head(5)
    for idx, row in translated_samples.iterrows():
        lang = row[lang_column].upper()
        original = row[text_column][:80]
        translated = row['text_translated'][:80]
        
        print(f"\n🌐 [{lang}] {original}...")
        print(f"🇬🇧 [EN]  {translated}...")
        print("-"*80)
    
    return df

In [38]:
cleaned_df = detect_languages(cleaned_df)
final_df = translate_non_english(cleaned_df)
final_df.head()

🔍 Detecting languages...


Detecting languages: 100%|██████████| 10233/10233 [00:33<00:00, 305.77it/s]



📊 Language Distribution:
   • en: 8209 (80.2%)
   • es: 344 (3.4%)
   • pt: 239 (2.3%)
   • fr: 222 (2.2%)
   • de: 145 (1.4%)
   • he: 143 (1.4%)
   • id: 136 (1.3%)
   • it: 123 (1.2%)
   • th: 64 (0.6%)
   • ja: 57 (0.6%)

⚠️  Non-English tweets: 2024 (19.8%)

🌐 Translating non-English tweets...
   Translating 2022 tweets...


Translating: 100%|██████████| 2022/2022 [45:28<00:00,  1.35s/it] 


✅ Translation complete!

📝 Translation Examples:

🌐 [FR] bien vue super les gars......
🇬🇧 [EN]  good view great guys......
--------------------------------------------------------------------------------

🌐 [AF] Super sleepy....
🇬🇧 [EN]  Super sleepy....
--------------------------------------------------------------------------------

🌐 [FR] Double super Fuckrat...
🇬🇧 [EN]  Double super Fuckrat...
--------------------------------------------------------------------------------

🌐 [PT] E PORCA BUTTANA MI TIRO UN SEGONE DUA LIPA ALL'OLIMPICO NEL 2026...
🇬🇧 [EN]  AND HOLY BITCH I'LL GET A HANDJOB WITH DUA LIPA AT THE OLIMPICO IN 2026...
--------------------------------------------------------------------------------

🌐 [FR] La team qui vendait la daube Astra Zeneca comme « super sûre » Changez rien les ...
🇬🇧 [EN]  The team that sold the Astra Zeneca crap as “super safe” Don’t change the champi...
--------------------------------------------------------------------------------


,id,text,created_at,username,likes,retweets,topic,timestamp,text_cleaned,text_translated,detected_lang
0,1960875468531621946,@PsyGuy007 bien vue super les gars...,2025-08-28,JaegerStep78492,0.0,NaN,Super Bowl LX,NaN,bien vue super les gars...,good view great guys...,fr
1,1960973143008292871,"You know what, I've ALSO been a splendid nigga...",2025-08-28,JSRingo,209.0,NaN,Super Bowl LX,NaN,"You know what, I've ALSO been a splendid nigga...","You know what, I've ALSO been a splendid nigga...",en
2,1961159089544990909,@fashion_nfl Don’t think enough people are tal...,2025-08-28,p23rce,479.0,NaN,Super Bowl LX,NaN,Don’t think enough people are talking about th...,Don’t think enough people are talking about th...,en
3,1960932935676387741,RT @tahsin_adib: NINGNING telling winter it mu...,2025-08-28,hidden_faayz,0.0,NaN,Winter Olympics 2026 Milano,NaN,NINGNING telling winter it must've take a long...,NINGNING telling winter it must've take a long...,en
5,1961003661548741029,Super sleepy. https://t.co/FMEc5ShIuv,2025-08-28,FajitaFantom,17.0,NaN,Super Bowl LX,NaN,Super sleepy.,Super sleepy.,af


## Save fianl dataset

In [ ]:
final_df.to_csv('final_trendingtopics.csv', index=False)